In [1]:
import pandas as pd

In [2]:
df=pd.read_csv('house_price.csv')

In [3]:
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   str    
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   str    
 6   Alley          91 non-null     str    
 7   LotShape       1460 non-null   str    
 8   LandContour    1460 non-null   str    
 9   Utilities      1460 non-null   str    
 10  LotConfig      1460 non-null   str    
 11  LandSlope      1460 non-null   str    
 12  Neighborhood   1460 non-null   str    
 13  Condition1     1460 non-null   str    
 14  Condition2     1460 non-null   str    
 15  BldgType       1460 non-null   str    
 16  HouseStyle     1460 non-null   str    
 17  OverallQual    1460 non-null   int64  
 18  OverallCond    1460

In [6]:
df.isnull().sum()[df.isnull().sum() > 0]

LotFrontage      259
Alley           1369
MasVnrType       872
MasVnrArea         8
BsmtQual          37
BsmtCond          37
BsmtExposure      38
BsmtFinType1      37
BsmtFinType2      38
Electrical         1
FireplaceQu      690
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
GarageCond        81
PoolQC          1453
Fence           1179
MiscFeature     1406
dtype: int64

In [10]:
df = df.drop(columns=['Alley', 'PoolQC', 'Fence', 'MiscFeature'])

In [11]:
df.isnull().sum()[df.isnull().sum() > 0]

LotFrontage     259
MasVnrType      872
MasVnrArea        8
BsmtQual         37
BsmtCond         37
BsmtExposure     38
BsmtFinType1     37
BsmtFinType2     38
Electrical        1
FireplaceQu     690
GarageType       81
GarageYrBlt      81
GarageFinish     81
GarageQual       81
GarageCond       81
dtype: int64

In [12]:
def toldir(df):
    for col in df.columns:
        if df[col].isnull().any():
            if df[col].dtype=='str':
                df[col]=df[col].fillna(df[col].mode()[0])
            else:
                df[col]=df[col].fillna(df[col].mean())
    return df

In [13]:
df=toldir(df)

In [16]:
df.isnull().sum()

Id               0
MSSubClass       0
MSZoning         0
LotFrontage      0
LotArea          0
                ..
MoSold           0
YrSold           0
SaleType         0
SaleCondition    0
SalePrice        0
Length: 77, dtype: int64

In [17]:
from sklearn.preprocessing import LabelEncoder
encoder=LabelEncoder()
def encodla(df):
    for col in df.columns:
        if df[col].dtype=='str':
            if df[col].nunique()<=3:
                dummies=pd.get_dummies(df[col],prefix=col,dtype=int)
                df=pd.concat([df.drop(columns=col),dummies],axis=1)
            else:
                df[col]=encoder.fit_transform(df[col])
    return df

In [18]:
df=encodla(df)
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,LotShape,LandContour,LotConfig,Neighborhood,Condition1,...,MasVnrType_BrkFace,MasVnrType_Stone,CentralAir_N,CentralAir_Y,GarageFinish_Fin,GarageFinish_RFn,GarageFinish_Unf,PavedDrive_N,PavedDrive_P,PavedDrive_Y
0,1,60,3,65.0,8450,3,3,4,5,2,...,1,0,0,1,0,1,0,0,0,1
1,2,20,3,80.0,9600,3,3,2,24,1,...,1,0,0,1,0,1,0,0,0,1
2,3,60,3,68.0,11250,0,3,4,5,2,...,1,0,0,1,0,1,0,0,0,1
3,4,70,3,60.0,9550,0,3,0,6,2,...,1,0,0,1,0,0,1,0,0,1
4,5,60,3,84.0,14260,0,3,2,15,2,...,1,0,0,1,0,1,0,0,0,1


In [20]:
from sklearn.preprocessing import MinMaxScaler
scaler=MinMaxScaler()
def scale(df):
    num_col=df.select_dtypes(include=['int64','float64']).columns.drop('SalePrice')
    df[num_col]=scaler.fit_transform(df[num_col])
    return df

In [21]:
df=scale(df)
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,LotShape,LandContour,LotConfig,Neighborhood,Condition1,...,MasVnrType_BrkFace,MasVnrType_Stone,CentralAir_N,CentralAir_Y,GarageFinish_Fin,GarageFinish_RFn,GarageFinish_Unf,PavedDrive_N,PavedDrive_P,PavedDrive_Y
0,0.000000,0.235294,0.75,0.150685,0.033420,1.0,1.0,1.0,0.208333,0.250,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
1,0.000685,0.000000,0.75,0.202055,0.038795,1.0,1.0,0.5,1.000000,0.125,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
2,0.001371,0.235294,0.75,0.160959,0.046507,0.0,1.0,1.0,0.208333,0.250,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
3,0.002056,0.294118,0.75,0.133562,0.038561,0.0,1.0,0.0,0.250000,0.250,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
4,0.002742,0.235294,0.75,0.215753,0.060576,0.0,1.0,0.5,0.625000,0.250,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0


In [22]:
df['SalePrice']

0       208500
1       181500
2       223500
3       140000
4       250000
         ...  
1455    175000
1456    210000
1457    266500
1458    142125
1459    147500
Name: SalePrice, Length: 1460, dtype: int64

In [24]:
from sklearn.model_selection import train_test_split
x=df.drop('SalePrice',axis=1)
y=df['SalePrice']

In [25]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [26]:
from sklearn.linear_model import LinearRegression
lr=LinearRegression()
lr

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [28]:
lr.fit(x_train,y_train)
y_pred=lr.predict(x_test)
y_pred

array([149053.31829422, 323665.82726632, 114060.36581912, 179200.74942916,
       321681.1126216 ,  67323.48092441, 238563.78659555, 144272.1718893 ,
        70362.76770761, 152138.45193664, 152795.38323875, 111050.27814524,
        68711.88726755, 212096.60195805, 163946.32833533, 137544.9266736 ,
       213960.50448953, 119667.09638702, 116637.28298502, 233833.13671802,
       132474.95200363, 211111.58560926, 190237.7634607 , 122239.1868816 ,
       219726.09617192, 158464.24515313, 209801.76017648,  71388.61653274,
       164609.87970066, 195530.17318144, 161951.76501803, 257091.95830907,
       190971.73640568,  97547.80770532, 254511.83080831, 143639.45904612,
       130895.16892602, 215812.64190436, 287325.39693535,  93893.94861353,
       122848.61696797, 253262.83366706, 107595.8139392 , 320527.65428019,
       130906.93731224, 152237.22804597,  97750.07543899, 133382.17660062,
       381984.57343668, 132946.50479435, 111647.85379218, 222625.59037839,
       100068.9395314 , 3

In [30]:
from sklearn.metrics import mean_absolute_error,r2_score
mae=mean_absolute_error(y_test,y_pred)
r2=r2_score(y_test,y_pred)

In [31]:
print('mae=',mae)
print('r2=',r2)

mae= 21373.28220065416
r2= 0.8500950270735168


In [32]:
from sklearn.model_selection import KFold,cross_val_score


In [33]:
kf=KFold(n_splits=5,shuffle=True,random_state=42)
kf

KFold(n_splits=5, random_state=42, shuffle=True)

In [34]:
scores=cross_val_score(lr,x,y,cv=kf,scoring='r2')

In [35]:
print(scores)

[0.85009503 0.81257494 0.37627552 0.85528412 0.90199571]


In [36]:
for i in scores:
    print(i)

0.8500950270735168
0.8125749381209859
0.37627552351815663
0.8552841214795888
0.9019957064317244


In [38]:
import numpy as np

In [40]:
np.mean(scores)

np.float64(0.7592450633247945)

In [41]:
np.std(scores)

np.float64(0.19357928953136058)

In [42]:
from sklearn.metrics import make_scorer

In [43]:
mae=make_scorer(mean_absolute_error,greater_is_better=False)

In [44]:
scores=cross_val_score(lr,x,y,cv=kf,scoring=mae)

In [47]:
print(-scores.mean())

20625.575840130048
